In [3]:
import random 
import numpy as np
import glob
import cv2
from sklearn.utils import shuffle
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
import joblib

pssm_glob = 'psiblast/data/pssm/*.pssm'

In [4]:
def read_matrix_from_pssm_file(fname):
    nrows=0
    with open(fname) as txt_file:    # Count rows.
        for line in txt_file:
            l = line.strip().split()
            if (len(l) == 44):
                nrows+=1;
    matrix = np.zeros((nrows,20), dtype="int8")
    i=0
    with open(fname) as txt_file: # Read PSSM.
        for line in txt_file:
            l = line.strip().split()
            if (len(l) == 44):
                values = np.array(list(map(int, l[2:22])))
                matrix[i,:] = values
                i+=1
    return matrix

# LOAD PSSM Matrices into protein_descriptors
protein_descritors = {}
for file in glob.glob(pssm_glob):
    pssm_matrix = read_matrix_from_pssm_file(file)
    prot_name = file.split("\\")[1].replace(".pssm", "")
    protein_descritors[prot_name] = pssm_matrix 
    
print("PSSM Matrixes read:")
print(len(protein_descritors))
#joblib.dump(protein_descritors,"protein_descritors.pkl", compress=3)

PSSM Matrixes read:
0


In [5]:
protein_descritors = joblib.load("protein_descritors.pkl")

In [6]:
# Create expected outputs array from sec_strucure file DSSP
dataset_file = open("sec_structs.txt", "r")
protein_secundary_structure = {}
protein_aa = {}
x = 0
name = ""
for line in dataset_file:
    if (x%4==0):
        prot_name = line.strip()
    elif(x%4==1):
        protein_aa[prot_name] = line.strip()
    elif(x%4==3):
        protein_secundary_structure[prot_name] = line.strip()
    x+=1

proteins = list(protein_secundary_structure.keys())
print(len(proteins))

1461


In [7]:
# for each proteins, for each amino extracts PSSM window with AA centered.
from scipy.ndimage import shift

X_protein_features = {}

protein_index = 0
for protein in proteins:
    pssm = protein_descritors[protein]
    protein_features = np.zeros((len(pssm),20,13))

    translated_pssm = pssm.T
    offset_start=6
    #amino_class = []

    ix = 0
    for AA in range(0, len(pssm)):
        pssmmatrix = translated_pssm[:,AA+offset_start-6:AA+offset_start-6+13]
        pssmmatrix= shift(pssmmatrix, (0,offset_start), cval=0)
        if (len(pssmmatrix.T) <13):
            ncols = 13-len(pssmmatrix.T)
            aux = np.zeros((20,ncols), dtype="int8")
            pssmmatrix = np.hstack((pssmmatrix,aux))

        protein_features[ix] = pssmmatrix
        #amino_class.append(y_proteins[protein_index])
       
        if (offset_start >0):
            offset_start-=1
        ix+=1
   
    X_protein_features[protein] = protein_features
    protein_index +=1

In [8]:
print(len(proteins))
#remove proteins with wrong Structural information
for prot in proteins:
    if (len(X_protein_features[prot]) != len(protein_secundary_structure[prot])):
        protein_secundary_structure.pop(prot)
    if (len(X_protein_features[prot])<30):
        protein_secundary_structure.pop(prot)
        
proteins = list(protein_secundary_structure.keys())
print(len(proteins))


        

1461
1427


In [9]:
# for each proteins, for each amino extracts PSSM window with AA centered.
from scipy.ndimage import shift

X_protein_features = {}

protein_index = 0
for protein in proteins:
    pssm = protein_descritors[protein]
    protein_features = np.zeros((len(pssm),20,13))

    translated_pssm = pssm.T
    offset_start=6
    #amino_class = []

    ix = 0
    for AA in range(0, len(pssm)):
        pssmmatrix = translated_pssm[:,AA+offset_start-6:AA+offset_start-6+13]
        pssmmatrix= shift(pssmmatrix, (0,offset_start), cval=0)
        if (len(pssmmatrix.T) <13):
            ncols = 13-len(pssmmatrix.T)
            aux = np.zeros((20,ncols), dtype="int8")
            pssmmatrix = np.hstack((pssmmatrix,aux))

        protein_features[ix] = pssmmatrix
        #amino_class.append(y_proteins[protein_index])
       
        if (offset_start >0):
            offset_start-=1
        ix+=1
   
    X_protein_features[protein] = protein_features
    protein_index +=1

In [10]:
proteins = proteins[:100]

In [11]:
from sklearn.model_selection import train_test_split

proteins_train, proteins_test = train_test_split(proteins, test_size=0.30, random_state=42)


In [12]:
y_train = []
for prot_name in proteins_train:
    for S in protein_secundary_structure[prot_name]:
        y_train.append(S)
print(len(y_train))

y_test = []
for prot_name in proteins_test:
    for S in protein_secundary_structure[prot_name]:
        y_test.append(S)
print(len(y_test))

X_train = np.zeros((len(y_train), 20,13))
aa_ix = 0
for prot in proteins_train:
    for AA in X_protein_features[prot]:
        X_train[aa_ix] = AA
        aa_ix += 1

X_test = np.zeros((len(y_test), 20,13))
aa_ix = 0
for prot in proteins_test:
    for AA in X_protein_features[prot]:
        X_test[aa_ix] = AA
        aa_ix += 1
        
print(X_train.shape)
print(X_test.shape)

19296
4868
(19296, 20, 13)
(4868, 20, 13)


In [13]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(y_train)
y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense, Flatten

model = Sequential()
model.add(LSTM(200, activation='relu', input_shape=(20, 13), return_sequences=True))
model.add(LSTM(800, activation='relu', return_sequences=False))

model.add(Flatten()) #Flatten the matrix to get it ready for dense.
model.add(Dense(100, activation='relu'))
model.add(Dense(3, activation='softmax'))

model.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()



Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 20, 200)           171200    
                                                                 
 lstm_3 (LSTM)               (None, 800)               3203200   
                                                                 
 flatten_1 (Flatten)         (None, 800)               0         
                                                                 
 dense_2 (Dense)             (None, 100)               80100     
                                                                 
 dense_3 (Dense)             (None, 3)                 303       
                                                                 
Total params: 3,454,803
Trainable params: 3,454,803
Non-trainable params: 0
_________________________________________________________________


In [15]:
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

#X_train, X_test, y_train, y_test = train_test_split(X, to_categorical(np.array(y)), test_size=0.33, random_state=42)

history = model.fit(X_train,
                         to_categorical(np.array(list(y_train))),
                         batch_size = 128,
                         verbose = 1,
                         epochs = 5,      #Changed to 3 from 50 for testing purposes.
                         validation_split = 0.2,
                         shuffle = False
                      #   callbacks=callbacks
                   )

print("Test_Accuracy: {:.2f}%".format(model.evaluate(X_test, to_categorical(np.array(list(y_test))) )[1]*100))


Epoch 1/5
121/121 [==============================] - 40s 321ms/step - loss: 0.9939 - accuracy: 0.5021 - val_loss: 0.9223 - val_accuracy: 0.5119
Epoch 2/5
121/121 [==============================] - 38s 318ms/step - loss: 0.9669 - accuracy: 0.5035 - val_loss: 0.9085 - val_accuracy: 0.5119
Epoch 3/5
121/121 [==============================] - 37s 309ms/step - loss: 0.9703 - accuracy: 0.5246 - val_loss: 0.9556 - val_accuracy: 0.5119
Epoch 4/5
121/121 [==============================] - 39s 323ms/step - loss: 0.9621 - accuracy: 0.5264 - val_loss: 0.8801 - val_accuracy: 0.5119
Epoch 5/5
153/153 [==============================] - 6s 42ms/step - loss: 0.8481 - accuracy: 0.6196
Test_Accuracy: 61.96%


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense, Flatten

model = Sequential()
model.add(LSTM(400, activation='relu', input_shape=(20, 13), return_sequences=True))
model.add(LSTM(600, activation='relu', return_sequences=False))

model.add(Flatten()) #Flatten the matrix to get it ready for dense.
model.add(Dense(100, activation='relu'))
model.add(Dense(3, activation='softmax'))

model.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_12 (LSTM)              (None, 20, 400)           662400    
                                                                 
 lstm_13 (LSTM)              (None, 600)               2402400   
                                                                 
 flatten_6 (Flatten)         (None, 600)               0         
                                                                 
 dense_12 (Dense)            (None, 100)               60100     
                                                                 
 dense_13 (Dense)            (None, 3)                 303       
                                                                 
Total params: 3,125,203
Trainable params: 3,125,203
Non-trainable params: 0
_________________________________________________________________


In [25]:
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

#X_train, X_test, y_train, y_test = train_test_split(X, to_categorical(np.array(y)), test_size=0.33, random_state=42)

history = model.fit(X_train,
                         to_categorical(np.array(list(y_train))),
                         batch_size = 128,
                         verbose = 1,
                         epochs = 5,      #Changed to 3 from 50 for testing purposes.
                         validation_split = 0.2,
                         shuffle = False
                      #   callbacks=callbacks
                   )

print("Test_Accuracy: {:.2f}%".format(model.evaluate(X_test, to_categorical(np.array(list(y_test))) )[1]*100))

Epoch 1/5
121/121 [==============================] - 38s 301ms/step - loss: 0.9985 - accuracy: 0.4971 - val_loss: 0.9861 - val_accuracy: 0.5119
Epoch 2/5
121/121 [==============================] - 36s 298ms/step - loss: 0.9812 - accuracy: 0.5194 - val_loss: 0.9506 - val_accuracy: 0.5119
Epoch 3/5
121/121 [==============================] - 36s 297ms/step - loss: 0.9280 - accuracy: 0.5422 - val_loss: 0.8501 - val_accuracy: 0.5137
Epoch 4/5
121/121 [==============================] - 36s 301ms/step - loss: 0.9592 - accuracy: 0.5661 - val_loss: 0.8531 - val_accuracy: 0.5150
Epoch 5/5
153/153 [==============================] - 7s 45ms/step - loss: 0.7720 - accuracy: 0.6401
Test_Accuracy: 64.01%
